# 🎙️ Reciter Recognition — v1 (من الـ baseline للـ app)
تكملة لـ `reciter_baseline_colab`. الجديد هنا:

| | v0 | v1 |
|---|---|---|
| التصنيف | كل فولدر = قارئ (الحصري مجوّد ≠ الحصري 128kbps مجوّد) | **شخص** واحد لكل قارئ، وكل فولدر ليه بصمة خاصة |
| الضجيج | ضجيج + صدى ثابت للتقييم بس | تدريب على تشكيلة ضجيج/صدى/تليفون/قص + **LDA** |
| الموديل | ECAPA جاهز | ECAPA + **fine-tuning** على التلاوة (اختياري) |
| غير معروف | عتبة على قراء مخفيين (فولدرات) | عتبة على **أشخاص** مخفيين، ومعايرة على val |
| قرآن/محاضرة | — | Whisper + مطابقة مع نص المصحف |
| الـ app | — | Gradio: فيديو/يوتيوب + مقطع صوتي |

**قبل التشغيل:**
1. Runtime ← Change runtime type ← **GPU (T4)**.
2. ارفعي فولدر المشروع (`quraa/` + `app.py`) على الدرايف في `MyDrive/reciters/code/`.

## 1) Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install speechbrain soundfile faster-whisper gradio yt-dlp

In [ ]:
import sys, shutil, subprocess
from pathlib import Path

ROOT       = Path('/content/drive/MyDrive/reciters')
DRIVE_DATA = ROOT / 'data'
CODE       = ROOT / 'code'            # quraa/ + app.py
MODELS     = ROOT / 'models'          # gallery_v1.npz, ecapa_ft.pt  (the app reads from here)
CACHE      = ROOT / 'cache'           # embeddings, so re-runs are fast
for p in (MODELS, CACHE): p.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(CODE))
import os; os.environ['QURAA_MODELS'] = str(MODELS)

WORK = Path('/content/work'); WORK.mkdir(exist_ok=True)
LOCAL_DATA = WORK / 'data'
if not LOCAL_DATA.exists():
    shutil.copytree(DRIVE_DATA, LOCAL_DATA)
print(len(list(LOCAL_DATA.rglob('*.mp3'))), 'mp3 files')

## 2) تحويل لـ wav + ربط كل فولدر بالقارئ الحقيقي

In [ ]:
import numpy as np, pandas as pd
from concurrent.futures import ThreadPoolExecutor
from quraa import reciters

man = pd.read_csv(LOCAL_DATA / 'manifest.csv')
WAV = WORK / 'wav'

def to_wav(rel):
    rel = rel.replace('\\', '/')
    src = LOCAL_DATA / rel
    dst = WAV / Path(*Path(rel).parts[1:]).with_suffix('.wav')
    if not dst.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(src),
                        '-ac', '1', '-ar', '16000', str(dst)], check=True)
    return str(dst)

with ThreadPoolExecutor(8) as ex:
    man['wav'] = list(ex.map(to_wav, man['path']))

man['person'] = man['reciter'].map(reciters.person_of)
man['style']  = man['reciter'].map(reciters.style_of)

# راجعي الجدول ده: لو فيه ربط غلط عدّليه في quraa/reciters.py
mapping = (man.groupby('reciter').agg(person=('person', 'first'), style=('style', 'first'), n=('wav', 'size'))
              .assign(name=lambda d: d.person.map(reciters.display_name)).sort_values('person'))
print(man['reciter'].nunique(), 'folders ->', man['person'].nunique(), 'people')
mapping

## 3) الـ embeddings (نظيفة) — بتتحفظ على الدرايف

In [ ]:
import soundfile as sf, torch
from tqdm.auto import tqdm
from quraa.embedder import Embedder
from quraa.augment import augment, augment_fixed

SR = 16000
def load_wav(p, max_s=30):
    w, sr = sf.read(p, dtype='float32')
    assert sr == SR
    return (w if w.ndim == 1 else w.mean(1))[: int(max_s * SR)]

def embed_paths(emb, paths, fn=None, max_s=30, batch=32, seed=0):
    # fn(w, rng) -> degraded w. Sorted by length so each batch has similar sizes.
    lens = [sf.info(p).frames for p in paths]
    order = np.argsort(lens)
    out = np.zeros((len(paths), 192), np.float32)
    def prep(i):
        w = load_wav(paths[i], max_s)
        return w if fn is None else fn(w, np.random.default_rng((seed, int(i))))
    with ThreadPoolExecutor(4) as ex:
        for b in tqdm(range(0, len(order), batch), leave=False):
            idx = order[b:b + batch]
            out[idx] = emb(list(ex.map(prep, idx)), batch_size=batch)
    return out

def cached(name, f):
    p = CACHE / f'{name}.npy'
    if p.exists():
        return np.load(p)
    x = f(); np.save(p, x); return x

emb0 = Embedder(cache_dir=str(MODELS / 'pretrained'))
paths = man['wav'].tolist()
E = cached('emb_clean_v0', lambda: embed_paths(emb0, paths))
print(E.shape)

## 4) التقييم على مستوى الشخص
نفس الـ gallery بتاع v0 (متوسط آيات train لكل فولدر)، بس بنحسب الإجابة صح لو طلع **نفس الشخص**.

In [ ]:
from quraa.gallery import Gallery

split   = man['split'].values
labels  = man['reciter'].values
persons = man['person'].values
tr, va, te = (split == 'train'), (split == 'val'), (split == 'test')

def evaluate(g, E_rows, true_persons, k=3):
    P = g.score_people(E_rows)
    top = np.argsort(-P, 1)[:, :k]
    y = np.array([g.person_ids.index(p) if p in g.person_ids else -1 for p in true_persons])
    return (top[:, 0] == y).mean(), (top == y[:, None]).any(1).mean()

def report(g, sets, title=''):
    rows = {name: evaluate(g, Er, pr) for name, (Er, pr) in sets.items()}
    print(title.ljust(28), ' | '.join(f'{n}: {a:.1%} (top3 {b:.1%})' for n, (a, b) in rows.items()))
    return rows

g0 = Gallery.build(E[tr], labels[tr])
_ = report(g0, {'val': (E[va], persons[va]), 'test': (E[te], persons[te])}, 'v0 gallery, person-level')

## 5) Embeddings بعد التشويه (ضجيج/صدى/تليفون/قص)
- `train`: نسختين مشوّهتين عشوائيًا لكل آية (للتدريب).
- `val/test`: نسخة مشوّهة عشوائيًا + التشويه الثابت بتاع v0 (علشان نقارن بـ 73.3%).

In [ ]:
rand_aug  = lambda w, rng: augment(w, rng)
fixed_aug = lambda w, rng: augment_fixed(w, rng)

def aug_sets(emb, tag):
    it = np.where(tr)[0]; iv = np.where(va | te)[0]
    A1 = cached(f'{tag}_train_aug1', lambda: embed_paths(emb, [paths[i] for i in it], rand_aug, max_s=15, seed=1))
    A2 = cached(f'{tag}_train_aug2', lambda: embed_paths(emb, [paths[i] for i in it], rand_aug, max_s=15, seed=2))
    R  = cached(f'{tag}_evl_rand',   lambda: embed_paths(emb, [paths[i] for i in iv], rand_aug, seed=3))
    F  = cached(f'{tag}_evl_fixed',  lambda: embed_paths(emb, [paths[i] for i in iv], fixed_aug, seed=4))
    E_rand = np.zeros_like(E); E_fixed = np.zeros_like(E)
    E_rand[iv] = R; E_fixed[iv] = F
    return it, np.vstack([A1, A2]), E_rand, E_fixed

it, A_tr, E_rand, E_fixed = aug_sets(emb0, 'v0')
print('train augmented:', A_tr.shape)

## 6) مقارنة الطرق
- **A**: بصمة من الآيات النظيفة بس (v0).
- **B**: بصمة من النظيف + المشوّه.
- **C**: B + إسقاط LDA متدرّب على النظيف + المشوّه (بيتعلم يتجاهل الغرفة/الميكروفون ويركّز على الصوت).

In [ ]:
def fit_methods(E_clean, A_tr, rows_tr, persons_ok=None):
    # rows_tr: indices (into man) of the clean train rows; A_tr = their augmented copies (x2).
    lab_tr = np.concatenate([labels[rows_tr]] * 3)
    per_tr = np.concatenate([persons[rows_tr]] * 3)
    X_tr   = np.vstack([E_clean[rows_tr], A_tr])
    keep = np.ones(len(X_tr), bool) if persons_ok is None else np.isin(per_tr, list(persons_ok))
    kc = keep[: len(rows_tr)]
    out = {'A': Gallery.build(E_clean[rows_tr][kc], labels[rows_tr][kc]),
           'B': Gallery.build(X_tr[keep], lab_tr[keep])}
    Xn = X_tr[keep] / np.linalg.norm(X_tr[keep], axis=1, keepdims=True)
    mean, W = Gallery.fit_lda(Xn, per_tr[keep])
    out['C'] = Gallery.build(X_tr[keep], lab_tr[keep], proj=(Xn.mean(0), W))
    return out

def compare(E_clean, A_tr, E_rand, E_fixed, title):
    gs = fit_methods(E_clean, A_tr, it)
    res = {}
    for k, g in gs.items():
        res[k] = report(g, {'val clean': (E_clean[va], persons[va]), 'val noisy': (E_rand[va], persons[va]),
                            'test clean': (E_clean[te], persons[te]), 'test v0-noise': (E_fixed[te], persons[te])},
                        f'{title} {k}')
    return gs, res

gs_v0, res_v0 = compare(E, A_tr, E_rand, E_fixed, 'pretrained')

## 7) Open-set: «قارئ غير معروف»
بنخبّي 20% من **الأشخاص** (مش الفولدرات)، والـ LDA بيتدرّب من غيرهم خالص.
العتبة بتتحدد على **val** (قبول 95% من المعروفين) وبنقيس على **test**.

In [ ]:
rng = np.random.default_rng(0)
people_all = sorted(set(persons))
unknown = set(rng.choice(people_all, size=max(1, len(people_all) // 5), replace=False))
known = [p for p in people_all if p not in unknown]
print('hidden people:', ', '.join(reciters.display_name(p) for p in sorted(unknown)))

def open_set(E_clean, A_tr, E_rand, method):
    g = fit_methods(E_clean, A_tr, it, persons_ok=known)[method]
    best = lambda X: g.score_people(X).max(1)
    kv = va & np.isin(persons, known)
    thr = np.percentile(np.concatenate([best(E_clean[kv]), best(E_rand[kv])]), 5)
    out = {'threshold': float(thr)}
    for name, X in [('clean', E_clean), ('noisy', E_rand)]:
        k = te & np.isin(persons, known); u = te & np.isin(persons, list(unknown))
        out[f'{name}: known accepted'] = (best(X[k]) >= thr).mean()
        out[f'{name}: unknown rejected'] = (best(X[u]) < thr).mean()
        out[f'{name}: top-1 known'] = evaluate(g, X[k], persons[k])[0]
    return out

open_v0 = {m: open_set(E, A_tr, E_rand, m) for m in 'ABC'}
pd.DataFrame(open_v0).T.style.format('{:.3f}')

## 8) Fine-tuning لـ ECAPA على التلاوة (اختياري — حوالي 15-25 دقيقة على T4)
- التدريب على **المعروفين بس** (نفس تقسيمة الخطوة 7) علشان نقدر نقيس غير المعروفين بأمانة.
- كل مثال: 3 ثواني عشوائية + تشويه عشوائي (80%) + ساعات صوت قارئ تاني في الخلفية.
- بنحفظ أحسن epoch حسب دقة val (نظيف + مشوّه).

In [ ]:
RUN_FINETUNE = True
FT_PATH = MODELS / 'ecapa_ft.pt'
EPOCHS = 8
WORKERS = 2

from quraa import finetune

# small fixed eval set for the per-epoch check (fast)
rs = np.random.default_rng(5)
_ev = np.where(va & np.isin(persons, known))[0]
ev_rows = np.array(sorted(rs.choice(_ev, min(400, len(_ev)), replace=False)))
cen_rows = np.concatenate([rs.choice(np.where(tr & (labels == l))[0], min(30, (tr & (labels == l)).sum()), replace=False)
                           for l in sorted(set(labels[np.isin(persons, known)]))])
ev_clean = [load_wav(paths[i], 8) for i in ev_rows]
ev_noisy = [augment(w, np.random.default_rng((9, j))) for j, w in enumerate(ev_clean)]
cen_wavs = [load_wav(paths[i], 8) for i in cen_rows]

def eval_fn(emb):
    g = Gallery.build(emb(cen_wavs), labels[cen_rows])
    a = evaluate(g, emb(ev_clean), persons[ev_rows])[0]
    b = evaluate(g, emb(ev_noisy), persons[ev_rows])[0]
    return (a + b) / 2

if RUN_FINETUNE:
    emb_ft = Embedder(cache_dir=str(MODELS / 'pretrained'))
    rows = np.where(tr & np.isin(persons, known))[0]
    pid = {p: i for i, p in enumerate(known)}
    hist = finetune.train(emb_ft, [paths[i] for i in rows], [pid[persons[i]] for i in rows],
                          FT_PATH, epochs=EPOCHS, workers=WORKERS, eval_fn=eval_fn)

In [ ]:
if RUN_FINETUNE:
    E_ft = cached('emb_clean_ft', lambda: embed_paths(emb_ft, paths))
    it, A_tr_ft, E_rand_ft, E_fixed_ft = aug_sets(emb_ft, 'ft')
    gs_ft, res_ft = compare(E_ft, A_tr_ft, E_rand_ft, E_fixed_ft, 'fine-tuned')
    open_ft = {m: open_set(E_ft, A_tr_ft, E_rand_ft, m) for m in 'ABC'}
    display(pd.DataFrame(open_ft).T.style.format('{:.3f}'))

> ⚠️ لو غيّرتي `EPOCHS` أو إعدادات التدريب، امسحي `cache/ft_*.npy` و`cache/emb_clean_ft.npy` علشان الـ embeddings تتحسب من جديد.

## 9) اختيار الأحسن وبناء القاعدة النهائية
المعيار: متوسط دقة **val نظيف + val مشوّه** (من غير ما نبص على test).
القاعدة النهائية بتستخدم **كل** الآيات (train+val+test) وكل القراء، والعتبة من تجربة open-set.

In [ ]:
cands = {('pretrained', m): (res_v0[m], open_v0[m]) for m in 'ABC'}
if RUN_FINETUNE:
    cands.update({('fine-tuned', m): (res_ft[m], open_ft[m]) for m in 'ABC'})
score = lambda r: (r['val clean'][0] + r['val noisy'][0]) / 2
(model_kind, method), (res_best, open_best) = max(cands.items(), key=lambda kv: score(kv[1][0]))
print('chosen:', model_kind, method, '| val score', f'{score(res_best):.1%}')
# لو عايزة تختاري بنفسك:  model_kind, method = 'fine-tuned', 'C'

emb_best = emb_ft if model_kind == 'fine-tuned' else emb0
E_b, tag = (E_ft, 'ft') if model_kind == 'fine-tuned' else (E, 'v0')

# final gallery: all rows clean + augmented copies of all rows (so val/test ayat help too)
all_rows = np.arange(len(man))
A_all = cached(f'{tag}_all_aug1', lambda: embed_paths(emb_best, paths, rand_aug, max_s=15, seed=11))
X = np.vstack([E_b, A_all]); L = np.concatenate([labels, labels]); Pp = np.concatenate([persons, persons])
if method == 'A':
    g_final = Gallery.build(E_b, labels)
elif method == 'B':
    g_final = Gallery.build(X, L)
else:
    Xn = X / np.linalg.norm(X, axis=1, keepdims=True)
    g_final = Gallery.build(X, L, proj=(Xn.mean(0), Gallery.fit_lda(Xn, Pp)[1]))

g_final.threshold = open_best['threshold']
g_final.meta = {'model': model_kind, 'method': method,
                'finetuned': 'ecapa_ft.pt' if model_kind == 'fine-tuned' else None,
                'val': {k: list(map(float, v)) for k, v in res_best.items()},
                'open_set': {k: float(v) for k, v in open_best.items()},
                'people': len(g_final.person_ids), 'folders': len(g_final.labels)}
g_final.save(MODELS / 'gallery_v1.npz')
print('saved', MODELS / 'gallery_v1.npz', '|', len(g_final.person_ids), 'people,', len(g_final.labels), 'recording sets')

## 10) قرآن ولا محاضرة؟
Whisper بيكتب اللي اتقال، وبنقيس نسبة الـ 3-grams اللي موجودة حرفيًا في المصحف.
- **إيجابي:** مقاطع ~30 ثانية من آيات test (متوصّلة ورا بعض) لقراء مختلفين.
- **سلبي:** محاضرات/خطب/بودكاست — حطي روابط في `LECTURE_URLS` أو ملفات في `reciters/lectures/`.

In [ ]:
from quraa.content import ContentDetector
from quraa import audio as qa, quran_text

det = ContentDetector()      # large-v3-turbo on GPU
print('whisper:', det.model_size)

LECTURE_URLS = [
    # 'https://www.youtube.com/watch?v=...',   # محاضرة / خطبة / درس
]
LECTURE_DIR = ROOT / 'lectures'

def chunks_of(w, n=30 * SR, k=4):
    starts = np.linspace(0, max(0, len(w) - n), k).astype(int)
    return [w[s:s + n] for s in starts]

rows = []
rs = np.random.default_rng(7)
for person in rs.choice(people_all, min(12, len(people_all)), replace=False):
    ids = np.where(te & (persons == person))[0]
    w = np.concatenate([load_wav(paths[i]) for i in rs.permutation(ids)[:8]])[: 30 * SR]
    r = det.classify_chunk(w); rows.append({'kind': 'quran', 'src': person, **r})

lec_sources = LECTURE_URLS + (sorted(map(str, LECTURE_DIR.glob('*'))) if LECTURE_DIR.exists() else [])
for src in lec_sources:
    w, name = qa.fetch(src, max_minutes=20)
    for c in chunks_of(w):
        r = det.classify_chunk(c); rows.append({'kind': 'lecture', 'src': name, **r})

cd = pd.DataFrame(rows)
display(cd[['kind', 'src', 'label', 'words', 'bigram', 'trigram', 'text']])
print(cd.groupby('kind')['trigram'].describe())
if (cd.kind == 'lecture').any():
    acc = ((cd.kind == 'quran') == (cd.label == 'quran')).mean()
    print(f'chunk accuracy at threshold {det.quran_threshold}: {acc:.1%}')
else:
    print('⚠️ مفيش محاضرات للاختبار — ضيفي LECTURE_URLS علشان نتأكد من العتبة.')

## 11) تجربة على فيديوهات حقيقية (الـ pipeline الكامل)
حطي فيديوهات/روابط وإنتي عارفة القارئ، علشان نقيس الأداء في الواقع مش في الاستوديو.

In [ ]:
from quraa.pipeline import Quraa
from quraa.report import to_markdown
from IPython.display import Markdown

engine = Quraa(MODELS / 'gallery_v1.npz')
engine._content = det           # reuse the loaded Whisper

REAL_TESTS = [
    ('/content/WhatsApp Video 2026-09-24 at 1.47.50 PM.mp4', 'minshawy'),
    # ('https://www.youtube.com/watch?v=...', 'husary'),
]
results = []
for src, truth in REAL_TESTS:
    if not qa.is_url(src) and not Path(src).exists():
        print('skip (missing):', src); continue
    r = engine.analyze(src, check_content=True)
    display(Markdown(to_markdown(r)))
    rec = r.get('reciter') or {}
    results.append({'src': src, 'truth': truth, 'pred': rec.get('person'), 'unknown': rec.get('unknown'),
                    'ok': rec.get('person') == truth and not rec.get('unknown')})
pd.DataFrame(results)

## 12) تشغيل الـ app (رابط عام من كولاب)

In [ ]:
from app import launch
launch(engine, share=True, debug=True)

### الخطوات الجاية
- **داتا من الواقع:** 20-50 فيديو يوتيوب معروف القارئ (مساجد، تراويح، تسجيلات موبايل) للاختبار — ده أهم مقياس.
- **قراء أكتر:** `collect_reciters_data.py --max-reciters 0` بيجيب كل قراء everyayah، وبعدين نعيد الخطوات 3-9.
- **قارئ مش في everyayah:** `Gallery.enroll(label, embeddings)` من 5-10 دقايق تلاوة نظيفة ليه.
- **النشر:** Hugging Face Spaces (Gradio) أو سيرفر فيه GPU — فولدر `models/` + الكود بس.